In [1]:
# Проверка задания п2.1.3 на устройстве: спрашивает у платы карту памяти командой mem_info
# и записывает обмен в файл device-2-1-3.log.

import time
from datetime import datetime

import serial
from serial.tools import list_ports

VENDOR_ID = 0x2E8A
PRODUCT_ID = 0x000A

TASK = "2.1.3"
PROJECT = "211-command-usb"
LOG_NAME = "device-2-1-3.log"
COMMANDS = ["mem_info"]
ANSWER_S = 2


def find_board():
    for port in list_ports.comports():
        if port.vid == VENDOR_ID and port.pid == PRODUCT_ID:
            return port
    return None


def talk(board):
    exchange = []
    with serial.Serial(board.device, timeout=0.5) as port:
        time.sleep(0.2)
        port.reset_input_buffer()
        started = time.monotonic()
        for command in COMMANDS:
            port.write((command + "\n").encode("ascii"))
            exchange.append((time.monotonic() - started, "-->", command))
            deadline = time.monotonic() + ANSWER_S
            while time.monotonic() < deadline:
                line = port.readline().decode("ascii", "replace").strip()
                if line:
                    exchange.append((time.monotonic() - started, "<--", line))
                    print(line, end="\r\n")
    return exchange


def write_log(board, exchange):
    with open(LOG_NAME, "w", encoding="utf-8") as log:
        log.write("задание: " + TASK + "\n")
        log.write("проект: " + PROJECT + "\n")
        log.write("устройство: %04x:%04x\n" % (board.vid, board.pid))
        log.write("серийный номер: " + str(board.serial_number) + "\n")
        log.write("порт: " + board.device + "\n")
        log.write("начало: " + datetime.now().isoformat(timespec="seconds") + "\n")
        for moment, direction, text in exchange:
            log.write("%8.3f %s %s\n" % (moment, direction, text))
        answers = len(exchange) - len(COMMANDS)
        log.write("итог: отправлено команд %d, принято строк %d\n" % (len(COMMANDS), answers))


board = find_board()

if board is None:
    print("Плата не найдена. Проверьте кабель и запишите на плату прошивку задания.", end="\r\n")
else:
    print("Плата на порту " + board.device + ", отправляю команды", end="\r\n")
    exchange = talk(board)
    write_log(board, exchange)
    print("Обмен записан в " + LOG_NAME, end="\r\n")


Плата на порту COM3, отправляю команды
mem_info
dbg read_line:110 got mem_info
area       start      end            size
flash      0x10000000 0x10200000  2097152
sram       0x20000000 0x20042000   270336
rom        0x00000000 0x00004000    16384
image      0x10000000 0x100082cc    33484
free       0x100082cc 0x10200000  2063668
boot2      0x10000000 0x10000100      256
text       0x10000100 0x1000734c    29260
data flash 0x1000734c 0x100082cc     3968
data ram   0x200000c0 0x20001040     3968
bss        0x20001040 0x20001c7c     3132
heap       0x20001c7c 0x20040000   254852
stack      0x20041800 0x20042000     2048
total
flash image       33484 = boot2 256 + text 29260 + data 3968
flash free    2063668 of 2097152
ram used           7100 = data 3968 + bss 3132
ram free         254852 for heap and 2048 for stack
Обмен записан в device-2-1-3.log
